# 02 - Data Wrangling
**Goal:** clean the raw API data and create the target label `Class` (1 = first stage landed successfully, 0 = it did not).

Input: `data/spacex_api_raw.csv` (from notebook 01). Output: `data/spacex_clean.csv`.

In [ ]:
!pip install -q pandas numpy

In [ ]:
import pandas as pd, numpy as np
pd.set_option('display.max_columns', None)
df = pd.read_csv('data/spacex_api_raw.csv')
print(df.shape)
df.head()

## Step 1 - Inspect data types and missing values

In [ ]:
print(df.dtypes)
print()
print((df.isnull().sum() / len(df) * 100).round(2).rename('% missing'))

## Step 2 - Remove duplicates and fix data types

In [ ]:
before = len(df)
df = df.drop_duplicates()
print('Duplicates removed:', before - len(df))
df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year
for col in ['GridFins', 'Reused', 'Legs']:
    df[col] = df[col].astype(bool)
df.dtypes

## Step 3 - Handle missing values
- `PayloadMass`: replace missing values with the **mean** (simple and common).
- `LandingPad`: missing means **no landing pad was used** (for example ocean landing), so we keep it as missing instead of filling it. We will not drop those rows.

In [ ]:
mean_payload = df['PayloadMass'].mean()
df['PayloadMass'] = df['PayloadMass'].fillna(mean_payload)
print('Mean payload used:', round(mean_payload, 2))
print(df.isnull().sum())

## Step 4 - Understand the launch sites, orbits and outcomes

In [ ]:
print(df['LaunchSite'].value_counts(), '\n')
print(df['Orbit'].value_counts(), '\n')
landing_outcomes = df['Outcome'].value_counts()
landing_outcomes

## Step 5 - Create the target label `Class`
In the API data, `Outcome` looks like `True ASDS`, `False Ocean`, `None None`, etc.
- `True ...` = landing succeeded
- `False ...` = landing failed
- `None ...` = no landing attempt

We treat success as **1** and everything else (failed or not attempted) as **0**.

In [ ]:
bad_outcomes = set(o for o in landing_outcomes.index if not o.startswith('True'))
print('Treated as unsuccessful (0):', bad_outcomes)
df['Class'] = df['Outcome'].apply(lambda x: 0 if x in bad_outcomes else 1)
print(df['Class'].value_counts())
print('Success rate:', round(df['Class'].mean() * 100, 2), '%')

In [ ]:
df.to_csv('data/spacex_clean.csv', index=False)
print('Saved data/spacex_clean.csv', df.shape)
df.head()

## Summary
We removed duplicates, fixed types, filled missing payload mass, and created the label `Class`.
The clean file is ready for EDA (notebook 03), SQL (notebook 04) and classification (notebook 06).